In [5]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, CODE], check=True)
print(subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
for _m in ["fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic"]:
    sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
414da19 flat-llrd-reversed



wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [6]:
EPOCHS = 10
ARMS = {
    "flat":          dict(lr_schedule="flat"),
    "flat_llrd":     dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="slow_deep"),
    "flat_llrd_rev": dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="fast_deep"),
}
# Every lr tried per arm (already-finished runs are skipped). Extend a list if its best is at an edge.
GRID = {
    "flat":          [0.01, 0.03, 0.1, 0.2, 0.3],
    "flat_llrd":     [0.01, 0.03, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9],
    "flat_llrd_rev": [0.03, 0.1, 0.2, 0.3, 0.5],   # new: 5 runs, ~45 GPU-min (estimate)
}

def grid_cfg(arm, lr):
    # The flat/0.03 point already exists as the epoch-sweep run; keeping its stage name
    # gives the identical config -> identical run_id -> run_sweep reuses that result.
    stage = "stage1_epoch_sweep" if (arm == "flat" and lr == 0.03) else "stage1_lr_grid"
    return sr.make_cfg(stage=stage, strategy="naive", epochs=EPOCHS, seed=0, lr=lr,
                       eval_split="val", **ARMS[arm])

CONFIGS = [grid_cfg(arm, lr) for arm in ARMS for lr in GRID[arm]]
SHOW = ["lr", "epochs", "eval_split"]
for c in CONFIGS: print(run_id(c), sr.arm_label(c), {k: c[k] for k in SHOW})

8965e350a6ae flat {'lr': 0.01, 'epochs': 10, 'eval_split': 'val'}
714a54d5fe81 flat {'lr': 0.03, 'epochs': 10, 'eval_split': 'val'}
c5e3da3ce9a2 flat {'lr': 0.1, 'epochs': 10, 'eval_split': 'val'}
4de4baf075b1 flat {'lr': 0.2, 'epochs': 10, 'eval_split': 'val'}
29f660df34fc flat {'lr': 0.3, 'epochs': 10, 'eval_split': 'val'}
f6325928117a llrd_slow_deep {'lr': 0.01, 'epochs': 10, 'eval_split': 'val'}
e4a01a3faae5 llrd_slow_deep {'lr': 0.03, 'epochs': 10, 'eval_split': 'val'}
f13c2d5c3a20 llrd_slow_deep {'lr': 0.1, 'epochs': 10, 'eval_split': 'val'}
bae4518da4f0 llrd_slow_deep {'lr': 0.2, 'epochs': 10, 'eval_split': 'val'}
a19024458615 llrd_slow_deep {'lr': 0.3, 'epochs': 10, 'eval_split': 'val'}
a253aa36605d llrd_slow_deep {'lr': 0.5, 'epochs': 10, 'eval_split': 'val'}
f2e88aaeca70 llrd_slow_deep {'lr': 0.7, 'epochs': 10, 'eval_split': 'val'}
8a4425c0ff26 llrd_slow_deep {'lr': 0.9, 'epochs': 10, 'eval_split': 'val'}
77a72524beba llrd_fast_deep {'lr': 0.03, 'epochs': 10, 'eval_split': 'v

In [7]:
out = run_sweep(CONFIGS, sr.run_experiment, paths)

[1/18] 8965e350a6ae already finished, skipping
[2/18] 714a54d5fe81 already finished, skipping
[3/18] c5e3da3ce9a2 already finished, skipping
[4/18] 4de4baf075b1 already finished, skipping
[5/18] 29f660df34fc already finished, skipping
[6/18] f6325928117a already finished, skipping
[7/18] e4a01a3faae5 already finished, skipping
[8/18] f13c2d5c3a20 already finished, skipping
[9/18] bae4518da4f0 already finished, skipping
[10/18] a19024458615 already finished, skipping
[11/18] a253aa36605d already finished, skipping
[12/18] f2e88aaeca70 already finished, skipping
[13/18] 8a4425c0ff26 already finished, skipping
[14/18] 77a72524beba running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 64.40%  current 64.40%  train-loss 1.378
  exp 2/20: seen-avg 40.60%  current 81.20%  train-loss 1.203
  exp 3/20: seen-avg 17.87%  current 53.60%  train-loss 1.649
  exp 4/20: seen-avg 19.50%  current 78.00%  train-loss 1.224
  exp 5/20: seen-avg 11.52%  current 57.60%  train-loss 1.659
  exp 6/20: seen-avg 10.33%  current 62.00%  train-loss 1.522
  exp 7/20: seen-avg 10.91%  current 76.40%  train-loss 1.306
  exp 8/20: seen-avg 8.65%  current 69.20%  train-loss 1.311
  exp 9/20: seen-avg 7.16%  current 64.40%  train-loss 1.537
  exp 10/20: seen-avg 7.32%  current 73.20%  train-loss 1.331
  exp 11/20: seen-avg 6.11%  current 67.20%  train-loss 1.463
  exp 12/20: seen-avg 5.97%  current 71.60%  train-loss 1.396
  exp 13/20: seen-avg 4.52%  current 58.80%  train-loss 1.651
  exp 14/20: seen-avg 4.80%  current 67.20%  train-loss 1.390
  exp 15/20: seen-avg 4.27%  current 64.00%  train-loss 1.454
  exp 16/20: seen-avg 4.80%  current 76.80%  train-loss 1.265
  exp 17/2

acc_current_exp,▄█▁▇▂▃▇▅▄▆▄▆▂▄▄▇█▅▆▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[15/18] 218c83d693a4 running


...

acc_current_exp,▃█▃▇▁▄▅▆▅▇▅▁▅▇███▇▆█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[18/18] 324e0679f865 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 75.20%  current 75.20%  train-loss 0.910
  exp 2/20: seen-avg 45.20%  current 90.40%  train-loss 0.500
  exp 3/20: seen-avg 22.80%  current 68.40%  train-loss 0.823
  exp 4/20: seen-avg 22.00%  current 88.00%  train-loss 0.513
  exp 5/20: seen-avg 14.80%  current 74.00%  train-loss 0.824
  exp 6/20: seen-avg 13.07%  current 78.40%  train-loss 0.726
  exp 7/20: seen-avg 11.66%  current 81.60%  train-loss 0.600
  exp 8/20: seen-avg 10.15%  current 81.20%  train-loss 0.451
  exp 9/20: seen-avg 8.89%  current 80.00%  train-loss 0.592
  exp 10/20: seen-avg 8.56%  current 85.60%  train-loss 0.537
  exp 11/20: seen-avg 7.05%  current 77.60%  train-loss 0.561
  exp 12/20: seen-avg 5.70%  current 68.40%  train-loss 0.689
  exp 13/20: seen-avg 6.00%  current 78.00%  train-loss 0.750
  exp 14/20: seen-avg 6.00%  current 84.00%  train-loss 0.487
  exp 15/20: seen-avg 5.41%  current 81.20%  train-loss 0.618
  exp 16/20: seen-avg 5.50%  current 88.00%  train-loss 0.501
  exp 17/

acc_current_exp,▃█▁▇▃▄▅▅▅▆▄▁▄▆▅▇▇▆▆▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


In [8]:
import numpy as np, pandas as pd
rows = []
for cfg in CONFIGS:
    r = load_result(cfg, paths)
    if r is None:
        rows.append({**{k: cfg[k] for k in SHOW}, "status": "not finished"}); continue
    m = r["metrics"]
    rows.append({"arm": sr.arm_label(cfg), **{k: cfg[k] for k in SHOW}, "run_id": r["run_id"],
                 "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
                 "avg_incremental_acc": m["avg_incremental_acc"],
                 # mean of the diagonal: how well each task was learned when it was current.
                 # Report next to forgetting: forgetting = peak - final, so it grows with this.
                 "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
                 "steps": m["optimizer_steps"], "n_params": m["n_params"], "minutes": r["wall_clock_s"] / 60})
df = pd.DataFrame(rows); df

,arm,lr,epochs,eval_split,run_id,final_avg_acc,avg_forgetting,avg_incremental_acc,acc_just_learned,steps,n_params,minutes
0,flat,0.01,10,val,8965e350a6ae,3.16,66.568421,12.122676,66.40,14200,1109240,8.982242
1,flat,0.03,10,val,714a54d5fe81,3.90,75.978947,13.468379,76.08,14200,1109240,117.254350
2,flat,0.10,10,val,c5e3da3ce9a2,4.44,81.368421,14.474445,81.74,14200,1109240,8.707282
3,flat,0.20,10,val,4de4baf075b1,4.56,83.305263,14.409329,83.70,14200,1109240,8.877636
4,flat,0.30,10,val,29f660df34fc,4.40,82.463158,14.247975,82.74,14200,1109240,8.244928
5,llrd_slow_deep,0.01,10,val,f6325928117a,2.84,56.084211,9.944344,56.12,14200,1109240,8.466168
6,llrd_slow_deep,0.03,10,val,e4a01a3faae5,3.60,68.547368,12.299203,68.72,14200,1109240,8.473640
7,llrd_slow_deep,0.10,10,val,f13c2d5c3a20,3.96,77.031579,13.628372,77.14,14200,1109240,8.565495
8,llrd_slow_deep,0.20,10,val,bae4518da4f0,4.00,79.305263,13.658882,79.34,14200,1109240,8.233691
9,llrd_slow_deep,0.30,10,val,a19024458615,4.24,80.757895,14.045176,80.96,14200,1109240,8.209625


In [9]:
LABEL = {sr.arm_label(sr.make_cfg(**a)): name for name, a in ARMS.items()}   # table label -> ARMS key
best = df.loc[df.groupby("arm")["avg_incremental_acc"].idxmax(), ["arm", "lr", "avg_incremental_acc", "acc_just_learned"]]
print(best)
for _, r in best.iterrows():
    lrs = GRID[LABEL[r["arm"]]]
    if r["lr"] in (min(lrs), max(lrs)):
        print(f"WARNING: best lr for {r['arm']} is at the edge of its grid ({r['lr']}); extend it before choosing.")

               arm   lr  avg_incremental_acc  acc_just_learned
2             flat  0.1            14.474445             81.74
17  llrd_fast_deep  0.5            14.333412             81.20
10  llrd_slow_deep  0.5            14.468723             82.08
